In [ ]:
from IPython.display import display, HTML

with open("style.css", "r") as f:
    css = f.read()

display(HTML(f"<style>{css}</style>"))

# Control analysis


In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
from functools import partial

def filler(num_noint: int, low: int, high:int, num_negs: int) -> np.ndarray:
    """Mix zeros and negative U[low, high] draws, then shuffle."""
    return np.random.permutation(np.concatenate([
        np.zeros(num_noint),
        -np.random.uniform(low, high, num_negs),
    ]))


def generate_interacs(n_specs = 4, k = None, method = "boosting", p_noint = 0):
    if method not in {"boosting", "cascade", "magnitude"}:
        raise ValueError(f"Unknown method '{method}'. Choose from: boosting, cascade, magnitude.")
    # ── Shared setup ──────────────────────────────────────────────────────────
    node_names = [f"S{i}" for i in range(1, n_specs + 1)]
    off_diag   = ~np.eye(n_specs, dtype=bool)
    M = np.full((n_specs, n_specs), np.nan)
    np.fill_diagonal(M, -0.5)
    # ── Method-specific fill ──────────────────────────────────────────────────
    if method == "boosting":
        total     = n_specs * (n_specs - 1)          # all off-diagonal cells
        num_noint = int(p_noint * total)
        num_negs = total - num_noint
        # Fill values
        M[off_diag] = filler(num_noint = num_noint, num_negs = num_negs, low=0.2, high=1)
        M[:, k]    *= 10                              # boost focal column
    elif method == "cascade":
        total     = (n_specs - 1) ** 2               # excludes k's own row
        num_noint = int(p_noint * total)
        num_negs  = total - num_noint
        # Cascade chain: col k feeds row 1, row 1 feeds row 2, … (skipping k)
        whom_rows = np.delete(np.arange(n_specs), k)
        who_cols  = np.concatenate([[k], whom_rows[:-1]])
        M[whom_rows, who_cols] = -20
        fill_mask  = off_diag.copy()
        fill_mask[whom_rows, who_cols] = False         # exclude cascade edges
        M[fill_mask] = filler(num_noint = num_noint, num_negs = num_negs, low=0.2, high=1)
        edges_to_highlight = pd.DataFrame({
            "from": [f"S{s}" for s in who_cols + 1],
            "to":   [f"S{t}" for t in whom_rows + 1],
        })
    elif method == "magnitude":  # magnitude
        total     = (n_specs - 1) ** 2
        num_noint = int(p_noint * total)
        num_negs  = total - num_noint
        fill_mask   = off_diag & (np.arange(n_specs) != k)   # off-diag, not col k
        M[fill_mask] = filler(num_noint=num_noint, num_negs=num_negs, low=1, high=2)  # U[-2, -1]
        M[:, k]      = -np.random.uniform(0.1, 1, n_specs)    # col k gets U[-1, 0]
        np.fill_diagonal(M, -0.5)                             # restore diagonal
    if method == 'boosting' or method == 'magnitude':
        other_nodes = [f"S{i}" for i in np.delete(np.arange(1, n_specs + 1), k)]
        edges_to_highlight = pd.DataFrame({"from": f"S{k + 1}", "to": other_nodes})
    # Round dataframe
    M_df = pd.DataFrame(np.round(M, 5), index=node_names, columns=node_names)
    return M_df, edges_to_highlight


# ── Style constants ───────────────────────────────────────────────────────────
STYLE = {
    "node_focal":    "#E07B39",  # warm amber
    "node_default":  "#4A7FB5",  # steel blue
    "node_border":   "#FFFFFF",  # white rim
    "edge_normal":   "#9BA8B5",  # muted slate
    "edge_hi":       "#C0392B",  # deep crimson
    "node_size":     1100,
    "label_color":   "#FFFFFF",
    "label_size":    11,
    "fig_bg":        "#F8F9FA",
}

def make_line(color, lw, label):
    return mlines.Line2D([], [], color=color, linewidth=lw, label=label,  solid_capstyle="round")

_METHODS = [
    ("boosting",  "Ejemplo de red método de incremento de columna",
     r"$\mathcal{U}(-1, 0)$",        r"$\mathcal{U}(-1, 0) \times 10$"),
    ("cascade",   "Ejemplo de red método de cascada",
     r"$\mathcal{U}(-1, 0)$",        r"$\mathcal{U}(-1, 0) \times 10$"),
    ("magnitude", "Ejemplo de red método de diferencia de magnitud",
     r"$\mathcal{U}(-2, -1)$",       r"$\mathcal{U}(-1, 0)$"),
]

METHOD_CONFIG = [
    {
        "fn":      partial(generate_interacs, method=method),
        "title":   title,
        "handles": [
            make_line(STYLE["edge_normal"], 1.5, label_normal),
            make_line(STYLE["edge_hi"],     3.5, label_hi),
        ],
    }
    for method, title, label_normal, label_hi in _METHODS
]

def make_line(color, lw, label):
    return mlines.Line2D([], [], color=color, linewidth=lw, label=label, solid_capstyle="round")

def draw_network(ax, M_df, edges_to_highlight, k, title, legend_handles):
    highlight_node  = f"S{k + 1}"
    highlight_pairs = set(zip(edges_to_highlight["from"], edges_to_highlight["to"]))
    G   = nx.from_pandas_adjacency(M_df, create_using=nx.DiGraph())
    pos = nx.circular_layout(G)
    node_colors  = [STYLE["node_focal"] if n == highlight_node  else STYLE["node_default"] for n in G.nodes()]
    hi_edges     = [(u, v) for u, v in G.edges() if (u, v) in highlight_pairs and u != v]
    normal_edges = [(u, v) for u, v in G.edges() if (u, v) not in highlight_pairs]
    # Nodes — white rim gives depth
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=STYLE["node_size"], edgecolors=STYLE["node_border"], linewidths=2.5)
    # Labels — white on colored nodes
    nx.draw_networkx_labels(G, pos, ax=ax, font_color=STYLE["label_color"], font_size=STYLE["label_size"], font_weight="bold")
    # Edges — curved to separate bidirectional pairs
    edge_defaults = dict(ax=ax, arrows=True, arrowsize=18, connectionstyle="arc3,rad=0.15", min_source_margin=28, min_target_margin=28)
    nx.draw_networkx_edges(G, pos, edgelist=normal_edges, edge_color=STYLE["edge_normal"], width=1.5, **edge_defaults)
    nx.draw_networkx_edges(G, pos, edgelist=hi_edges, edge_color=STYLE["edge_hi"],    width=3.0, **edge_defaults)
    # Title and legend
    ax.set_title(title, fontsize=12, fontweight="bold", pad=14, color="#2C3E50")
    ax.axis("off")
    legend = ax.legend(handles=legend_handles, loc="upper right",
                       fontsize=9, frameon=True, framealpha=0.92,
                       edgecolor="#DEE2E6", fancybox=False)
    legend.get_frame().set_linewidth(0.8)


# ── Figure setup ──────────────────────────────────────────────────────────────
plt.close("all")
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
fig.patch.set_facecolor(STYLE["fig_bg"])

for ax, cfg in zip(axes, METHOD_CONFIG):
    ax.set_facecolor(STYLE["fig_bg"])
    n_specs = 5
    k = 3
    p_noint=0.1
    M_df, edges_to_highlight = cfg["fn"](n_specs = n_specs, k =k, p_noint=p_noint)
    draw_network(ax, M_df, edges_to_highlight, k, cfg["title"], cfg["handles"])

fig.suptitle("Redes de interacción: métodos de incremento, cascada y magnitud", fontsize=15, fontweight="bold", color="#2C3E50")

plt.tight_layout()
plt.savefig('/mnt/data/sur/users/mrivera/thesis_plots/panel_controls.png', dpi = 300)
plt.show()

# Control proposal analysis

Based on the methods previously described, one thousand simulations were performed to analyze the efficiency of generating data where a species from the community acts as a keystone. Three methods were tested:

1. Keystone species column boosting
In silico communities were generated, where one species (*k*) was selected as keystone. Its outward interactions were boosted by a constant of 10 to ensure that the species held the largest interactions in the network.


2.  Keystone species column with difference of magnitude
In silico communities were generated, where one species (*k*) was selected as keystone. Its outward interactions were constrained to be smaller than all other available interactions in the network. The rationale was that removal of the keystone species would result in an increase in the interactions of all remaining species — by definition, since keystone interactions were sampled from U(−1, 0) while all other interactions were sampled from U(−1, −2) or set to 0.

3. Cascade effect of the network
To generate the interaction matrix, one species (*k*) was selected to initiate the cascade chain of effects. Interactions belonging to this chain were sampled from a uniform distribution *U*(−1, 0) and subsequently amplified by a constant of 10, ensuring they represented the largest-magnitude interactions within the network. For all remaining off-diagonal interactions outside the chain it was sampled from a U(−1, 0]
 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd
from pathlib import Path
from matplotlib.patches import Patch

def significance_label(p):
    if p < 0.0001: return '****'
    elif p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else: return 'ns'

def plotter(alive_data, all_data, plot_title):
    cols_plot  = ['keystoneness', 'dissimilarity_bc', 'prop_extinctions']
    all_row  = ['Keystoneness (todas las esp.)',   'Disimilitud BC (todas las esp.)',   'Prop. Extinciones (todas las esp.)']
    surv_row = ['Keystoneness (esp. sobreviv.)',   'Disimilitud BC (esp. sobreviv.)',   'Prop. Extinciones (esp. sobreviv.)']
    group_labels = ['No escencial', 'Escencial']
    data = [all_data, alive_data]
    palette = {False: '#5B8DB8', True: '#C97070'}
    plt.close('all')
    fig, axes = plt.subplots(2, len(cols_plot), figsize=(18,9), sharey=False)
    fig.suptitle(plot_title, fontsize=15, fontweight='bold')
    # Define styling
    BOXPLOT_STYLE = dict(
        patch_artist=True,
        positions=[1, 2],
        widths=0.5,
        medianprops=dict(color='black', linewidth=2),
        whiskerprops=dict(color='black', linewidth=1.2),
        capprops=dict(color='red', linewidth=2),
        flierprops=dict(visible=False),
        boxprops=dict(linewidth=1.2)
    )
    # Plot
    for row_idx, (ax_row, ax_data) in enumerate(zip(axes, data)):
        col_labels = all_row if row_idx == 0 else surv_row
        for ax, col, col_label in zip(ax_row, cols_plot, col_labels):
            groups = [
                np.log1p(ax_data[ax_data['is_keystone'] == k][col].dropna())
                for k in [False, True]
            ]
            # Statistical test
            _, p = stats.mannwhitneyu(groups[0], groups[1], alternative='two-sided')
            # Boxplot
            bp = ax.boxplot(groups, **BOXPLOT_STYLE)
            for patch, k in zip(bp['boxes'], [False, True]):
                patch.set_facecolor(palette[k])
                patch.set_alpha(0.7)
            # Jittered points
            # for j, (group, k) in enumerate(zip(groups, [False, True])):
            #     jitter = np.random.normal(j + 1, 0.07, size=len(group))
            #    ax.scatter(jitter, group, alpha=0.2, s=5, color=palette[k], zorder=2)
            # Significance annotation
            ax.text(1.5, max(g.max() for g in groups), significance_label(p), ha='center', fontsize=13, fontweight='bold')
            # Style
            ax.set_facecolor('#F0F0F0')
            ax.yaxis.grid(True, color='white', linewidth=1.2)
            ax.set_axisbelow(True)
            ax.spines[:].set_visible(False)
            ax.set_title(col_label, fontweight='bold', fontsize=12, pad=8)
            ax.set_xticks([1, 2])
            ax.set_xticklabels(group_labels, fontsize=10)
            ax.tick_params(axis='both', length=0)
            ax.set_ylabel('Valor de la métrica (escala log1p)', fontsize=11)
    # Shared legend
    handles = [plt.Rectangle((0,0),1,1, color=palette[k], alpha=0.7) for k in [False, True]]
    fig.legend(handles, group_labels, loc='lower center', ncol=2,frameon=False, fontsize=10, bbox_to_anchor=(0.5, -0.04))
    return fig


def process_file(param, data_dir):
    f = f"{data_dir}/ExtSummaries/ExtSummary_{param['id']}.feather"
    x = pd.read_feather(f)[['rel_pop_initial', 'prop_extinctions', 'dissimilarity_bc', 'keystoneness']]
    x['is_keystone'] = False
    x.loc[param['key'] - 1, 'is_keystone'] = True
    return x

In [ ]:
# Wrapper for all methods
from IPython.display import display

def wrapper(data_dir, fig_name, title):
    params = pd.read_csv(f'{data_dir}/simulation_params.tsv', sep = '\t')
    all_summary = pd.concat(
        [process_file(param = params.iloc[i], data_dir=data_dir) for i in range(len(params))],
        ignore_index=True
    )
    alive_summary = all_summary[all_summary['rel_pop_initial'] > 1e-06]
    figure = plotter(alive_data=alive_summary, all_data=all_summary, plot_title = title)
    figure.tight_layout()
    figure.savefig(f'/mnt/data/sur/users/mrivera/thesis_plots/{fig_name}', dpi=300)
    display(figure)   # ← not figure.show(

setup = [
    ('/mnt/data/sur/users/mrivera/Data/Controls/Boosted_keystone',  'panel_boosted.png',    'Distribuciones del método incremento de columna'),
    ('/mnt/data/sur/users/mrivera/Data/Controls/Cascade_keystone',  'panel_cascade.png',    'Distribuciones del método de cascada'),
    ('/mnt/data/sur/users/mrivera/Data/clean_controls/Negative_magnitude_d1e776c23c74','panel_magnitude.png',  'Distribuciones del método de magnitud'),
]

plt.clf()
plt.close('all')
for data_dir, fig_name, title in setup:
    wrapper(data_dir, fig_name, title)
    print(f'>> Finished panel for {fig_name}\n')


## Caption
**Figure 1.** Distribution of the metrics used to quantify the impact of species extinction on the community. The top row shows metric distributions computed across all species in the community, while the bottom row is restricted to species that survived above the extinction threshold (relative population > 1×10⁻⁶). In each panel, boxplots compare keystone and non-keystone species. Asterisks denote the significance level of pairwise comparisons evaluated with the Mann–Whitney U test.

# Interpretation
Across all three methods, the median distributions of keystoneness and Bray-Curtis dissimilarity remained consistently centered near zero. This is explained by the low relative abundance of most species at the time of perturbation, rendering their removal largely indistinguishable from background variation given their negligible individual effect on community dynamics.

Based on the control analysis, we conclude that none of the three methods reliably produced *in silico* communities in which a single species exhibited clear keystone behavior. Nevertheless, the column boosting method was selected as the proof of concept for the GNN, as it yielded the most distinct and statistically significant keystoneness distributions among the approaches evaluated. It is worth noting that the proportion of extinctions did not differ visually across methods and was therefore excluded as a prediction target; keystoneness was selected instead as prediction target.